# Data Preparation - `wellness.csv`

In [1]:
# Import libraries
import math
import numpy as np
import pandas as pd
import seaborn as sns
import scipy.stats as stats
import matplotlib.pyplot as plt

from collections import defaultdict
from sklearn.preprocessing import MinMaxScaler

import seaborn as sns
from scipy.stats import pearsonr


In [2]:
# Loading Datasets
patients_df = pd.read_csv('../datasets/patients.csv')
wellness_df = pd.read_csv('../datasets/patients_wellness.csv', low_memory=False)

len_init_wellness = len(wellness_df)

In [3]:
wellness_df

,Unnamed: 0,subject_id,BloodPressure,Weight,BMI,Height,BloodPressureStanding1,BloodPressureLying,BloodPressureSitting,BloodPressureStanding3,BloodPressureStanding,Weight2,eGFR
0,0,10000032,110/65,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,10000084,NaN,94.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2,10000117,NaN,NaN,18.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3,10000248,NaN,NaN,NaN,60.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,4,10000280,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
158311,158311,19999636,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
158312,158312,19999733,120/74,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
158313,158313,19999750,NaN,NaN,28.2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
158314,158314,19999784,NaN,NaN,NaN,61.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## **Data Cleaning**


In [4]:
wellness_df = wellness_df.drop(columns=["Unnamed: 0", 'BloodPressureStanding3'])

In [5]:
wellness_df.columns

Index(['subject_id', 'BloodPressure', 'Weight', 'BMI', 'Height',
       'BloodPressureStanding1', 'BloodPressureLying', 'BloodPressureSitting',
       'BloodPressureStanding', 'Weight2', 'eGFR'],
      dtype='object')

In [6]:
wellness_df.head()

,subject_id,BloodPressure,Weight,BMI,Height,BloodPressureStanding1,BloodPressureLying,BloodPressureSitting,BloodPressureStanding,Weight2,eGFR
0,10000032,110/65,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,10000084,NaN,94.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,10000117,NaN,NaN,18.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,10000248,NaN,NaN,NaN,60.0,NaN,NaN,NaN,NaN,NaN,NaN
4,10000280,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
essential_subset_columns = ['BloodPressure', 'Weight', 'BMI', 'Height',
                            'BloodPressureStanding1', 'BloodPressureLying',
                            'BloodPressureSitting', 'BloodPressureStanding',
                            'Weight2', 'eGFR']
wellness_df.dropna(how='all', subset=essential_subset_columns, inplace=True)
len_remaining_wellness = len(wellness_df)
remaining_percentage = (len_remaining_wellness / len_init_wellness) * 100

print("Numero iniziale di righe:", len_init_wellness)
print("Numero di righe rimanenti:", len_remaining_wellness)
print(f"Percentuale di righe rimanenti: {remaining_percentage:.2f}%")


Numero iniziale di righe: 158316
Numero di righe rimanenti: 13725
Percentuale di righe rimanenti: 8.67%


In [8]:
patients_df.shape

(41415, 19)

In [9]:
wellness_df = wellness_df[wellness_df['subject_id'].isin(patients_df['subject_id'])]
print(len(wellness_df))

2639


In [10]:
percentage_missing = (len(wellness_df) / len_init_wellness) * 100
print(f"Percentuale di dati rimanenti post rimozione: {percentage_missing:.2f}%")


Percentuale di dati rimanenti post rimozione: 1.67%


In [11]:
wellness_df

,subject_id,BloodPressure,Weight,BMI,Height,BloodPressureStanding1,BloodPressureLying,BloodPressureSitting,BloodPressureStanding,Weight2,eGFR
0,10000032,110/65,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,10000117,NaN,NaN,18.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
41,10002528,NaN,NaN,24.4,NaN,NaN,NaN,NaN,NaN,NaN,NaN
43,10002557,NaN,170.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
55,10003019,NaN,NaN,NaN,64.5,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
158294,19998198,NaN,NaN,21.3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
158295,19998330,NaN,NaN,NaN,62.0,NaN,NaN,NaN,NaN,NaN,NaN
158296,19998350,NaN,116.62,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
158312,19999733,120/74,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [12]:
# Conta e calcola la percentuale di valori mancanti
missing_values = wellness_df.isnull().sum()
missing_percentage = (missing_values / len(wellness_df)) * 100
print("Totale presenti : ", len(wellness_df))
print(pd.DataFrame({'Missing val': missing_values, '    |     Missing %': missing_percentage, '    |   totale presenti nell\'attributo': len(wellness_df) - missing_values}))


Totale presenti :  2639
                        Missing val      |     Missing %  \
subject_id                        0             0.000000   
BloodPressure                  2001            75.824176   
Weight                         1939            73.474801   
BMI                            1982            75.104206   
Height                         2018            76.468359   
BloodPressureStanding1         2635            99.848427   
BloodPressureLying             2633            99.772641   
BloodPressureSitting           2629            99.621069   
BloodPressureStanding          2639           100.000000   
Weight2                        2638            99.962107   
eGFR                           2637            99.924214   

                            |   totale presenti nell'attributo  
subject_id                                                2639  
BloodPressure                                              638  
Weight                                                     7

In [13]:
wellness_df = wellness_df.drop(columns=['BloodPressureStanding'])

**Osservazione**: Calcolo del `BMI` dove non presente sfruttando li dove sono presenti sia `Height` e `Weight` nel `subject_id`

**Osservazione**: Riempio la colonna BloodPressure li dove NaN e' presente con valori presenti negli altri attributi BloodPressureX per creare un unico attributo e aumentare la percentuale di dati presenti per BloodPressure

In [14]:
# Filtra le righe dove 'Height', 'Weight' e 'BMI' non sono nulli
filtered_rows = wellness_df.dropna(subset=['BloodPressureSitting'])

# Visualizza il risultato
print(len(filtered_rows))
print(filtered_rows)


10
        subject_id BloodPressure  Weight  BMI  Height BloodPressureStanding1  \
3469      10224486           NaN     NaN  NaN     NaN                    NaN   
14407     10925041           NaN     NaN  NaN     NaN                    NaN   
17929     11146315           NaN     NaN  NaN     NaN                    NaN   
37535     12379978           NaN     NaN  NaN     NaN                    NaN   
56508     13590729           NaN     NaN  NaN     NaN                    NaN   
58530     13721078           NaN     NaN  NaN     NaN                    NaN   
71049     14512069           NaN     NaN  NaN     NaN                    NaN   
78623     14991456           NaN     NaN  NaN     NaN                    NaN   
87580     15554944           NaN     NaN  NaN     NaN                    NaN   
126762    18017189           NaN     NaN  NaN     NaN                    NaN   

       BloodPressureLying BloodPressureSitting  Weight2 eGFR  
3469                  NaN               144/87      N

In [15]:
# Copia i valori di 'BloodPressureSitting' in 'BloodPressure' solo se 'BloodPressure' è NaN
wellness_df['BloodPressure'] = np.where(
    wellness_df['BloodPressure'].isna(),  # Condizione: se 'BloodPressure' è NaN
    wellness_df['BloodPressureSitting'],   # Usa il valore di 'BloodPressureSitting'
    wellness_df['BloodPressure']           # Altrimenti mantieni il valore esistente
)

# Ora posso cancellare l'attributo `BloodPressureSitting`
wellness_df = wellness_df.drop(columns=['BloodPressureSitting'])



In [16]:
# Filtra le righe dove 'Height', 'Weight' e 'BMI' non sono nulli
filtered_rows = wellness_df.dropna(subset=['BloodPressureLying'])

# Visualizza il risultato
print(len(filtered_rows))
print(filtered_rows)


6
        subject_id BloodPressure  Weight  BMI  Height BloodPressureStanding1  \
13624     10876180           NaN     NaN  NaN     NaN                    NaN   
50851     13225691           NaN     NaN  NaN     NaN                    NaN   
129318    18179909           NaN     NaN  NaN     NaN                    NaN   
139722    18838669           NaN     NaN  NaN     NaN                    NaN   
146730    19272439           NaN     NaN  NaN     NaN                    NaN   
151411    19562282           NaN     NaN  NaN     NaN                    NaN   

       BloodPressureLying  Weight2 eGFR  
13624              109/47      NaN  NaN  
50851              144/90      NaN  NaN  
129318             112/76      NaN  NaN  
139722             105/65      NaN  NaN  
146730             120/50      NaN  NaN  
151411             112/78      NaN  NaN  


In [17]:
# Copia i valori di 'BloodPressureSitting' in 'BloodPressure' solo se 'BloodPressure' è NaN
wellness_df['BloodPressure'] = np.where(
    wellness_df['BloodPressure'].isna(),  # Condizione: se 'BloodPressure' è NaN
    wellness_df['BloodPressureLying'],   # Usa il valore di 'BloodPressureSitting'
    wellness_df['BloodPressure']           # Altrimenti mantieni il valore esistente
)

# Ora posso cancellare l'attributo `BloodPressureSitting`
wellness_df = wellness_df.drop(columns=['BloodPressureLying'])



In [18]:
# Filtra le righe dove 'Height', 'Weight' e 'BMI' non sono nulli
filtered_rows = wellness_df.dropna(subset=['BloodPressureStanding1'])

# Visualizza il risultato
print(len(filtered_rows))
print(filtered_rows)


4
        subject_id BloodPressure  Weight  BMI  Height BloodPressureStanding1  \
29713     11885997           NaN     NaN  NaN     NaN                 130/92   
71050     14512099           NaN     NaN  NaN     NaN                 110/65   
78624     14991553           NaN     NaN  NaN     NaN                 112/60   
134455    18509741           NaN     NaN  NaN     NaN                 110/68   

        Weight2 eGFR  
29713       NaN  NaN  
71050       NaN  NaN  
78624       NaN  NaN  
134455      NaN  NaN  


In [19]:
# Copia i valori di 'BloodPressureSitting' in 'BloodPressure' solo se 'BloodPressure' è NaN
wellness_df['BloodPressure'] = np.where(
    wellness_df['BloodPressure'].isna(),  # Condizione: se 'BloodPressure' è NaN
    wellness_df['BloodPressureStanding1'],   # Usa il valore di 'BloodPressureSitting'
    wellness_df['BloodPressure']           # Altrimenti mantieni il valore esistente
)

# Ora posso cancellare l'attributo `BloodPressureSitting`
wellness_df = wellness_df.drop(columns=['BloodPressureStanding1'])



In [20]:
wellness_df['BloodPressure'].isnull().sum()

1981

In [21]:
len(wellness_df) - wellness_df['BloodPressure'].isnull().sum()

658

In [22]:
# Filtra le righe dove 'Height', 'Weight' e 'BMI' non sono nulli
filtered_rows = wellness_df.dropna(subset=['Weight2'])

# Visualizza il risultato
print(len(filtered_rows))
print(filtered_rows)


1
       subject_id BloodPressure  Weight  BMI  Height  Weight2 eGFR
17688    11132535           NaN     NaN  NaN     NaN    130.0  NaN


In [23]:
wellness_df = wellness_df.drop(columns=['Weight2'])


In [24]:
# Filtra le righe dove 'Height', 'Weight' e 'BMI' non sono nulli
filtered_rows = wellness_df.dropna(subset=['eGFR'])

# Visualizza il risultato
print(len(filtered_rows))
print(filtered_rows)


2
       subject_id BloodPressure  Weight  BMI  Height  eGFR
89008    15641146           NaN     NaN  NaN     NaN   >60
98873    16262815           NaN     NaN  NaN     NaN  77.6


In [25]:
wellness_df = wellness_df.drop(columns=['eGFR'])


In [26]:
wellness_df

,subject_id,BloodPressure,Weight,BMI,Height
0,10000032,110/65,NaN,NaN,NaN
2,10000117,NaN,NaN,18.0,NaN
41,10002528,NaN,NaN,24.4,NaN
43,10002557,NaN,170.00,NaN,NaN
55,10003019,NaN,NaN,NaN,64.5
...,...,...,...,...,...
158294,19998198,NaN,NaN,21.3,NaN
158295,19998330,NaN,NaN,NaN,62.0
158296,19998350,NaN,116.62,NaN,NaN
158312,19999733,120/74,NaN,NaN,NaN


---
 
- `Weight` : $\text{Se } X \notin [30, 300] \Rightarrow X \text{ è un outlier}$
- `BMI`: $\text{Se } X \notin [12, 60] \Rightarrow X \text{ è un outlier}$
- `Height` : $\text{Se } X \notin [23.6, 98.4] \Rightarrow X \text{ è un outlier}$

**Osservazione**: gli outlier e valori mancanti verrano eliminati perche rischio di mettere dati fasulli per casi sensibili in cui le combinazioni potrebbero non tornare. Per cui chiedere al medico come procedere

In [27]:
# Statistiche per ogni NUMERICAL column per determinare anomalie
tr_tmp_column = ['Weight', 'BMI', 'Height', 'Weight2']
print(wellness_df.describe())

         subject_id       Weight        BMI      Height
count  2.639000e+03   700.000000  657.00000  621.000000
mean   1.505849e+07   177.672971   28.76621   65.801063
std    2.903212e+06    77.650433   12.67577    4.193037
min    1.000003e+07    79.890000   11.70000   54.000000
25%    1.249487e+07   141.000000   24.00000   63.000000
50%    1.510690e+07   167.700000   27.10000   66.000000
75%    1.755661e+07   203.000000   31.70000   68.500000
max    1.999978e+07  1741.900000  299.00000   97.700000


In [28]:
# Conta e calcola la percentuale di valori mancanti
missing_values = wellness_df.isnull().sum()
missing_percentage = (missing_values / len(wellness_df)) * 100
print(pd.DataFrame({'Missing val': missing_values, '    |     Missing %': missing_percentage}))


               Missing val      |     Missing %
subject_id               0             0.000000
BloodPressure         1981            75.066313
Weight                1939            73.474801
BMI                   1982            75.104206
Height                2018            76.468359
